In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Находим корень проекта (поднимаемся на уровень выше папки notebooks)
root = Path.cwd().parent
sys.path.append(str(root))


from lib.data.analysis_pipeline import (
    load_and_preprocess,
    create_epochs,
    save_epochs,
    plot_epochs_images,
)
from lib.data.raw_preprocessing import apply_notch_bandpass_car
import lib.data.config as data_config
CONFIG_PATH = (root / "patients.yaml").resolve().parent / "config.py"
data_config.initialize_config(CONFIG_PATH)
ch_to_keep = data_config.ch_to_keep
best_ch_by_power = data_config.best_ch_by_power
import torch
from omegaconf import OmegaConf

%matplotlib inline

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal, stats


import mne

from scipy.signal import stft
from intervaltree import Interval, IntervalTree
from tqdm import tqdm

import numpy as np
import matplotlib.pyplot as plt
from mne.filter import notch_filter
from scipy.signal import welch

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
import numpy as np
import mne

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

	
import abc
from sklearn.decomposition import PCA

In [ ]:
files_list = {}
root = "../../PirogovDATA/"
subject = "s11"

files_list[subject] = load_and_preprocess(root, subject, best_ch_by_power=best_ch_by_power, autoclean=True)

# Single

In [ ]:
# =======================
# Preprocessing: notch + bandpass -> global average reference (CAR)
# (place between load_and_preprocess(...) and create_epochs(...))
# =======================

clean = {subject: []}
sfreq = 2000  # пример

events_avg = [1, 2, 3, 4, 5, 6, 7, 8, 10]
event_single = [9]

# параметры (подстрой при необходимости)
notch_freqs = np.arange(50, sfreq // 2, 50)   # 50,100,150,... до Найквиста
l_freq, h_freq = 1.0, 150.0                   # bandpass

for i, raw in enumerate(files_list[subject]):
    r_car = apply_notch_bandpass_car(raw, notch_freqs=notch_freqs, l_freq=l_freq, h_freq=h_freq, method='iir', n_jobs=-1)

    # (опционально) если хочешь быстро проверить PSD после CAR
    # r_car.plot_psd(fmax=250); plt.show()

    clean[subject].append(r_car)

# дальше по твоему коду:
# epochs = create_epochs(clean, subject, event_id=[*events_avg, *event_single])[0]


In [ ]:
for i, data in enumerate(clean[subject]):
    data.plot_psd()
    plt.show()

In [ ]:
epochs = create_epochs(clean, subject, event_id=[*events_avg, *event_single], epoch_thresh_dict=epoch_thresh_dict)[0] # return epochs_list, event_dict
# [0.1, 1.3, ..., 120]
freqs = np.linspace(0.1, 120, 100)  
# For each frequency, a specific number of cycles 
# is specified – the ‘window length’ in cycles.
n_cycles = freqs / 2.0              
tfr = mne.time_frequency.tfr_morlet(
    epochs[0], freqs=freqs, n_cycles=n_cycles, return_itc=False, decim=2, average=False
)
tfr.save(f"../tfr_{subject}.fif")

# cycle

In [ ]:
root = "../../PirogovDATA/"

for subject in list(best_ch_by_power.keys()):
    files_list = {}

    files_list[subject] = load_and_preprocess(root, subject, best_ch_by_power=best_ch_by_power, autoclean=True)
    clean = {subject: []}
    sfreq = 2000  # пример

    events_avg = [1, 2, 3, 4, 5, 6, 7, 8, 10]
    event_single = [9]

    # параметры (подстрой при необходимости)
    notch_freqs = np.arange(50, sfreq // 2, 50)   # 50,100,150,... до Найквиста
    l_freq, h_freq = 1.0, 150.0                   # bandpass

    for i, raw in enumerate(files_list[subject]):
        # r = raw.copy().load_data()  # важно: фильтры требуют данных в памяти

        # 1) notch (сетевой шум и гармоники)
        # r.notch_filter(freqs=notch_freqs, n_jobs=-1)

        # 2) bandpass
        # r.filter(l_freq=l_freq, h_freq=h_freq, method='iir')

        # 3) global average reference (CAR): x_i(t) <- x_i(t) - mean_j x_j(t)
        # X = r.get_data()                         # (n_ch, n_times)
        # X = X - X.mean(axis=0, keepdims=True)    # вычитаем среднее по каналам

        # создаём RawArray с корректным info
        # info = r.info.copy()
        # r_car = mne.io.RawArray(X, info)

        # переносим аннотации (события)
        # r_car.set_annotations(r.annotations)

        r_car = apply_notch_bandpass_car(raw, notch_freqs=notch_freqs, l_freq=l_freq, h_freq=h_freq, method='iir', n_jobs=-1)

        # (опционально) если хочешь быстро проверить PSD после CAR
        # r_car.plot_psd(fmax=250); plt.show()

        clean[subject].append(r_car)

    # дальше по твоему коду:
    # epochs = create_epochs(clean, subject, event_id=[*events_avg, *event_single])[0]
    for i, data in enumerate(clean[subject]):
        print(f"subject {subject}, file #{i}")
        data.plot_psd()
        plt.show()

    epochs = create_epochs(clean, subject, event_id=[*events_avg, *event_single], epoch_thresh_dict=epoch_thresh_dict)[0] # return epochs_list, event_dict
    # [0.1, 1.3, ..., 120]
    freqs = np.linspace(0.1, 120, 100)  
    # For each frequency, a specific number of cycles 
    # is specified – the ‘window length’ in cycles.
    n_cycles = freqs / 2.0              
    tfr = mne.time_frequency.tfr_morlet(
        epochs[0], freqs=freqs, n_cycles=n_cycles, return_itc=False, decim=2, average=False
    )
    tfr.save(f"../specs_with_car/tfr_{subject}.fif")
    